In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [37]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [38]:
v1.dot(dv)

np.float32(0.32332397)

In [39]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [40]:
v2.dot(dv)

np.float32(0.019730574)

In [8]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [42]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [9]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [44]:

print(len(texts))

1350


In [10]:
from tqdm.auto import tqdm

batch_size = 50

vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

In [46]:
len(vectors)

1350

In [60]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

print(scores)
print(max(scores))

[np.float32(0.48740587), np.float32(0.2099193), np.float32(0.76294106), np.float32(0.44378525), np.float32(0.26083997), np.float32(0.4866516), np.float32(0.30061552), np.float32(0.56009984), np.float32(0.45960498), np.float32(0.25628167), np.float32(0.33153272), np.float32(0.27318522), np.float32(0.27691635), np.float32(0.34122986), np.float32(0.46007156), np.float32(0.26240277), np.float32(0.39200056), np.float32(0.10854157), np.float32(0.27567303), np.float32(0.16646828), np.float32(0.31437922), np.float32(0.0066854246), np.float32(0.11205028), np.float32(0.21905681), np.float32(0.3400085), np.float32(0.23571287), np.float32(0.32714826), np.float32(0.15088344), np.float32(0.16563272), np.float32(0.05545033), np.float32(0.23541182), np.float32(0.08533001), np.float32(-0.0030899644), np.float32(-0.042598374), np.float32(-0.060277097), np.float32(0.006491136), np.float32(0.034277484), np.float32(-0.049589746), np.float32(-0.0006707795), np.float32(-0.017013624), np.float32(0.07062715), 

In [11]:
import numpy as np

X = np.array(vectors)

print(X)

[[-0.02670618 -0.12245757  0.01594413 ... -0.00230654 -0.11218394
  -0.02365559]
 [-0.01099552 -0.11074744 -0.02536942 ...  0.09022228 -0.02697371
   0.01975672]
 [-0.08896548 -0.06128178  0.00775603 ...  0.0405971   0.00479277
  -0.02745943]
 ...
 [-0.03652925  0.01415426 -0.06838644 ...  0.04316786  0.08105537
  -0.02148626]
 [-0.13091588 -0.06990605 -0.0093188  ... -0.00044342 -0.0128573
   0.01426918]
 [-0.07984784  0.01926981  0.02544978 ... -0.03368027 -0.01884026
   0.05837054]]


In [62]:
scores = X.dot(v1)

In [63]:
scores

array([ 0.48740587,  0.2099193 ,  0.762941  , ..., -0.08637964,
        0.03759784, -0.03037035], shape=(1350,), dtype=float32)

In [64]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [65]:
documents[2]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [70]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [12]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [73]:
vindex.search(v1, num_results=5, filter_dict={'course':'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

In [13]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [14]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [15]:
query = 'I just found out about the program, can I still enroll?'
assistant.rag(query) 

'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [16]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self,query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict,
        )

In [17]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client = openai_client
)

In [18]:
vector_assistant.rag(query)

'Yes, but if you want a certificate, you need to submit your project while submissions are still open.'

In [19]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [20]:
vs_index.fit(vectors, documents)

In [23]:
query = 'I just found out about the course, can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

In [24]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [25]:
vs_index.close()